# Apport de la statistique inférentielle dans l'optimisation d'un classifieur SVM
## Application au diagnostic médical (cancer du sein)

**Master : Mathématiques et Ingénierie Numérique — Faculté des Sciences de Rabat**

Auteur : Doha Et-touimy — Encadrant : M. Mourad El Azhari

---
### Plan
1. Chargement et nettoyage des données
2. Analyse statistique des données (description, heatmap, droite de Henry)
3. Modèle SVM
4. Estimation et inférence (estimation ponctuelle, intervalle de confiance, MLE)
5. Validation statistique du modèle (Student, Z-test, khi-deux)
6. Conclusion

## 0. Importation des bibliothèques

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import ttest_ind, ttest_1samp, norm, chi2
from scipy.optimize import minimize

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Paramètres globaux
RANDOM_STATE = 42
ALPHA = 0.05        # seuil de signification
MU_0 = 14.0         # norme médicale
P_0 = 0.90          # standard médical
VAR = 'mean radius'

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (9, 6)

## 1. Chargement et nettoyage des données

Base *Breast Cancer Wisconsin* (UCI) : 569 observations, 30 variables explicatives quantitatives, 1 variable cible qualitative (0 = maligne, 1 = bénigne).

In [ ]:
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target     # 0 = Maligne, 1 = Bénigne

print('taille', df.shape)
df.head()

In [ ]:
# --- Nettoyage ---
print('Valeurs manquantes :', df.isnull().sum().sum())
print('Doublons           :', df.duplicated().sum())

df = df.drop_duplicates()
cols = df.select_dtypes(include='number').columns
df[cols] = df[cols].fillna(df[cols].median())

print('Taille après nettoyage :', df.shape)

## 2. Analyse statistique des données
### 2.1 Échantillonnage et description

In [ ]:
n = len(df)
print(f"Taille de l'échantillon (n) : {n}")
print(f"Tumeurs malignes (0) : {(df['target']==0).sum()}")
print(f"Tumeurs bénignes (1) : {(df['target']==1).sum()}")
df[VAR].describe()

### 2.2 Analyse exploratoire : matrice de corrélation (Heatmap)

In [ ]:
cols10 = df.columns[:10]
corr = df[cols10].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=.5, annot_kws={'size': 8})
plt.title('Matrice de Corrélation - Axe 1 (Analyse Exploratoire)')
plt.tight_layout()
plt.show()

# Couples fortement corrélés
for i in range(len(cols10)):
    for j in range(i+1, len(cols10)):
        if abs(corr.iloc[i, j]) > 0.9:
            print(f"{cols10[i]:22s} <-> {cols10[j]:22s} r = {corr.iloc[i,j]:.3f}")

Une corrélation $r \approx 1.00$ entre `mean radius`, `mean perimeter` et `mean area` indique une redondance importante de l'information et la présence de **multicolinéarité**. Cela justifie une réduction de dimension (ACP) afin d'améliorer les performances du SVM et de limiter le surapprentissage.

### 2.3 Vérification de la normalité : droite de Henry (Q-Q plot)

Équation de la droite théorique : $y = \dfrac{1}{\sigma}(x - \mu)$

In [ ]:
serie = df[VAR]

plt.figure(figsize=(9, 7))
stats.probplot(serie, dist='norm', plot=plt)
plt.title('Vérification de la Normalité : Droite de Henry')
plt.xlabel('Quantiles théoriques')
plt.ylabel('Valeurs observées (Mean Radius)')
plt.grid(True, linestyle=':')
plt.show()

W, p_sw = stats.shapiro(serie)
print(f"Shapiro-Wilk : W = {W:.4f}, p-value = {p_sw:.6f}")

Les points expérimentaux s'alignent de manière quasi linéaire sur la droite théorique : la distribution de `mean radius` suit **approximativement** une loi normale.

## 3. Modèle SVM

Hyperplan : $w \cdot x + b = 0$ — Décision : $y_{pred} = \mathrm{signe}(w \cdot x + b)$

Marge : $\dfrac{2}{\lVert w \rVert}$ — Optimisation : $\min_{w,b} \tfrac12 \lVert w \rVert^2$ sous $y_i(w \cdot x_i + b) \ge 1$

In [ ]:
X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_svm = SVC(kernel='linear', random_state=RANDOM_STATE)
model_svm.fit(X_train_scaled, y_train)

y_pred = model_svm.predict(X_test_scaled)
f_accuracy = accuracy_score(y_test, y_pred)
n_test = len(y_test)

print(f"Accuracy (f) : {f_accuracy*100:.2f}%   (n_test = {n_test})")
print('\nMatrice de confusion :\n', confusion_matrix(y_test, y_pred))
print('\n', classification_report(y_test, y_pred, target_names=['Maligne','Bénigne']))

### 3.1 Visualisation de la frontière de décision (via ACP)

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_train_scaled)

svm_2d = SVC(kernel='linear', random_state=RANDOM_STATE).fit(X_2d, y_train)

h = .05
x_min, x_max = X_2d[:,0].min()-1, X_2d[:,0].max()+1
y_min, y_max = X_2d[:,1].min()-1, X_2d[:,1].max()+1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(9, 7))
plt.contourf(xx, yy, Z, alpha=.25, cmap='coolwarm')
for c, col, lab in [(0,'royalblue','Maligne (0)'), (1,'crimson','Bénigne (1)')]:
    m = np.asarray(y_train) == c
    plt.scatter(X_2d[m,0], X_2d[m,1], c=col, edgecolor='k', s=40, label=lab)
plt.title('Frontière de Décision du Classifieur SVM (Visualisation 2D via PCA)')
plt.xlabel('Composante Principale 1 (PCA1)')
plt.ylabel('Composante Principale 2 (PCA2)')
plt.legend()
plt.show()

## 4. Estimation et inférence
### 4.1 Estimation ponctuelle des paramètres ($\mu$, $\sigma$)

$\hat{\mu} = \bar{x}$ (estimateur sans biais) et $\hat{\sigma} = \sigma_e \sqrt{\dfrac{n}{n-1}}$

In [ ]:
m_e = df[VAR].mean()
sigma_e = df[VAR].std(ddof=0)
sigma = df[VAR].std(ddof=1)
sigma_corrige = sigma_e * np.sqrt(n/(n-1))

print(f"Moyenne observée (m_e)          : {m_e:.2f}")
print(f"Écart-type observé (sigma_e)    : {sigma_e:.2f}")
print(f"Écart-type estimé sans biais    : {sigma_corrige:.2f}  (ddof=1 : {sigma:.2f})")

### 4.2 Intervalle de confiance de la performance du SVM

Condition de validité : $n \ge 30 \Rightarrow$ Théorème Central Limite applicable.

$$IC = \left[f - t_\alpha\sqrt{\frac{f(1-f)}{n}} \; ; \; f + t_\alpha\sqrt{\frac{f(1-f)}{n}}\right]$$

In [ ]:
t_alpha = 1.96
erreur_standard = np.sqrt((f_accuracy * (1 - f_accuracy)) / n_test)
ic_inf = max(0, f_accuracy - t_alpha * erreur_standard)
ic_sup = min(1, f_accuracy + t_alpha * erreur_standard)

print('resultats')
print(f"Accuracy (f): {f_accuracy*100:.2f}%")
print(f"Intervalle de Confiance (95%): [{ic_inf*100:.2f}% , {ic_sup*100:.2f}%]")

### 4.3 Maximum de Vraisemblance (MLE) appliqué au SVM

Les sorties du SVM sont transformées en probabilités via la sigmoïde :

$$\sigma(x) = \frac{1}{1 + e^{-(ax+b)}}$$

Les paramètres $a$ et $b$ sont estimés en maximisant la log-vraisemblance :

$$\log L(a,b) = \sum_i \left[ y_i \log p_i + (1-y_i)\log(1-p_i) \right]$$

In [ ]:
distances_train = model_svm.decision_function(X_train_scaled)
distances = model_svm.decision_function(X_test_scaled)

def neg_log_vraisemblance(params, d, y):
    a, b = params
    z = a * d + b
    log_p = -np.logaddexp(0, -z)      # log(sigma(z))
    log_q = -np.logaddexp(0, z)       # log(1 - sigma(z))
    return -np.sum(y * log_p + (1 - y) * log_q)

res = minimize(neg_log_vraisemblance, [1.0, 0.0],
               args=(distances_train, np.asarray(y_train)), method='BFGS')
a, b = res.x
print(f"Paramètres estimés par MLE : a = {a:.4f}, b = {b:.4f}")
print(f"Log-vraisemblance maximale : {-res.fun:.4f}")

In [ ]:
# Probabilités issues de la sigmoïde estimée par MLE
p_benigne = 1 / (1 + np.exp(-(a * distances + b)))
p_maligne = 1 - p_benigne

# Comparaison avec la calibration de Platt de scikit-learn
model_proba = SVC(kernel='linear', probability=True,
                  random_state=RANDOM_STATE).fit(X_train_scaled, y_train)
probs = model_proba.predict_proba(X_test_scaled)

results = pd.DataFrame({
    'Distance (Géométrique)': distances[:5],
    'Probabilité Maligne (via MLE)': p_maligne[:5],
    'Probabilité Bénigne (via MLE)': p_benigne[:5],
    'P. Bénigne (sklearn)': probs[:5, 1],
})
print(results.to_string())

## 5. Validation statistique du modèle
### 5.1 Test de Student d'homogénéité

$H_0 : \mu_M = \mu_B$ (le rayon moyen n'est pas discriminant)

$H_1 : \mu_M \neq \mu_B$ (le rayon moyen est significatif)

$$t_{obs} = \frac{\bar{x}_A - \bar{x}_B}{\sqrt{s_A^2/n_A + s_B^2/n_B}} \qquad |t_{obs}| > t_{\alpha/2} \Rightarrow \text{Rejet de } H_0$$

In [ ]:
group_M = df[df['target'] == 0][VAR]
group_B = df[df['target'] == 1][VAR]

t_obs, p_value = ttest_ind(group_M, group_B, equal_var=False)
print(f"1. Statistique t (t_obs) : {t_obs:.4f}")
print(f"2. Test de Student (p-value) : {p_value:.3e}")
print(f"   Valeur critique : t_alpha/2 = 1.96")

if p_value < ALPHA:
    print(f"-> Décision : Rejet de H0 (p < {ALPHA})")
    print("-> Conclusion : Le rayon moyen est une variable SIGNIFICATIVE.")
else:
    print(f"-> Décision : Non-rejet de H0 (p >= {ALPHA})")
    print("-> Conclusion : Le rayon moyen n'est pas une variable significative.")

### 5.2 Test de Student de conformité

$H_0 : \mu = 14.0$ (échantillon conforme à la norme) contre $H_1 : \mu \neq 14.0$

$$z_{obs} = \frac{\bar{x} - \mu_0}{s/\sqrt{n}}$$

In [ ]:
t_obs2, p_value2 = ttest_1samp(df[VAR], popmean=MU_0)

print(f"Norme de référence (mu0) : {MU_0}")
print(f"Moyenne de l'échantillon : {df[VAR].mean():.4f}")
print(f"Statistique t (t_obs)    : {t_obs2:.4f}")
print(f"p-value                  : {p_value2:.4f}")

if p_value2 < ALPHA:
    print(f"-> Décision : Rejet de H0 (p < {ALPHA})")
    print("-> Conclusion : L'écart est significatif par rapport à la norme.")
else:
    print(f"-> Décision : Non-rejet de H0 (p >= {ALPHA})")
    print("-> Conclusion : L'échantillon est conforme à la norme médicale.")

### 5.3 Test d'hypothèse sur une proportion (Z-test)

$H_0 : p = 0.90$ (performance conforme) contre $H_1 : p > 0.90$ (performance supérieure)

$$z_{obs} = \frac{f - p_0}{\sqrt{\dfrac{p_0(1-p_0)}{n}}}$$

In [ ]:
f = f_accuracy
z_obs = (f - P_0) / np.sqrt((P_0 * (1 - P_0)) / n_test)
p_value3 = 1 - norm.cdf(z_obs)

print(f"Paramètres : f = {f:.4f}, p0 = {P_0}, n = {n_test}")
print(f"Statistique z (z_obs) : {z_obs:.4f}")
print(f"Valeur critique       : z_alpha = 1.645")
print(f"p-value               : {p_value3:.4f}")

if z_obs > 1.645:
    print('Décision : Rejet de H0 (Le modèle est supérieur au standard médical)')
else:
    print('Décision : Non-rejet de H0 (performance conforme au standard)')

### 5.4 Test du Khi-deux d'adéquation

$H_0$ : le modèle est adéquat — $H_1$ : le modèle est inadéquat

$$\chi^2_{obs} = \sum_{i=1}^{k} \frac{(O_i - T_i)^2}{T_i} \qquad \chi^2_{obs} > \chi^2_{1-\alpha}(\nu) \Rightarrow \text{Rejet de } H_0$$

In [ ]:
classes = np.unique(np.concatenate([y_test, y_pred]))
obs_counts = np.array([(np.asarray(y_pred) == c).sum() for c in classes], float)
exp_counts = np.array([(np.asarray(y_test) == c).sum() for c in classes], float)

chi2_obs = np.sum((obs_counts - exp_counts)**2 / exp_counts)
ddl = len(classes) - 1
chi2_critique = chi2.ppf(1 - ALPHA, ddl)     # = 3.84 pour ddl = 1
p_value4 = 1 - chi2.cdf(chi2_obs, ddl)

print("======= TEST DU KHI-DEUX D'ADÉQUATION =======")
print(f"Effectifs réels (Ti)   : {exp_counts.astype(int).tolist()}")
print(f"Effectifs prédits (Oi) : {obs_counts.astype(int).tolist()}")
print('-' * 45)
print(f"Statistique Khi2 (obs) : {chi2_obs:.4f}")
print(f"p-value                : {p_value4:.4f}")

if chi2_obs > chi2_critique:
    print(f"\n-> Décision : Rejet de H0 (Car {chi2_obs:.3f} > {chi2_critique:.2f})")
    print('-> Conclusion : Le modèle est INADÉQUAT (différence significative).')
else:
    print(f"\n-> Décision : Non-rejet de H0 (Car {chi2_obs:.3f} <= {chi2_critique:.2f})")
    print('-> Conclusion : Le modèle est ADÉQUAT (bonne adéquation).')

## 6. Conclusion

Cette étude montre que la performance d'un modèle SVM en diagnostic médical doit être **validée par la statistique inférentielle** :

- le **maximum de vraisemblance** permet une interprétation probabiliste des sorties du classifieur, indispensable en contexte médical ;
- le **test de Student d'homogénéité** confirme que la variable `mean radius` est fortement discriminante ;
- le **test de conformité** montre que l'échantillon est représentatif des standards médicaux ;
- le **Z-test sur une proportion** établit que la précision du modèle dépasse significativement le standard de 90 % ;
- le **test du khi-deux** confirme la bonne adéquation entre les prédictions et la distribution réelle.

Le SVM constitue ainsi un outil de diagnostic **robuste et scientifiquement fiable**, dont les conclusions peuvent être généralisées de l'échantillon à la population.